# Stage 3 — Text Preprocessing

**Goal:** turn raw `title` + `abstract` text into clean, normalized tokens ready for feature engineering (TF-IDF) in Stage 4.

### Pipeline for this stage
1. Combine title + abstract into one text field
2. Strip LaTeX/math artifacts
3. Lowercase, remove punctuation/numbers
4. Tokenize
5. Remove stopwords
6. Lemmatize
7. Sanity-check before/after on real examples
8. Save cleaned data for Stage 4

### Why combine title + abstract?
Titles are short but often contain the clearest topic signal (e.g. "Mamba Networks for X"). Abstracts add depth/context but can be noisier. Combining both gives the vectorizer more signal per document without needing a separate model for each field. This is a documented design decision — an alternative would be to weight title terms more heavily, which is worth mentioning as a limitation/future improvement in your report.

In [1]:
import pandas as pd
import re
import nltk

# One-time downloads for NLTK's resources (safe to re-run, it skips if already present)
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("punkt")
nltk.download("punkt_tab")

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ndahiro_jonathan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/ndahiro_jonathan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/ndahiro_jonathan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/ndahiro_jonathan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ndahiro_jonathan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
df = pd.read_parquet("../data/processed/arxiv_filtered.parquet")
df.shape

(30000, 5)

In [3]:
# Step 1: combine title + abstract into one field
df["text"] = df["title"] + ". " + df["abstract"]
df["text"].iloc[0][:300]  # peek at the first 300 characters of the first document

'Multi-Exit Kolmogorov-Arnold Networks: enhancing accuracy and parsimony. Kolmogorov-Arnold Networks (KANs) uniquely combine high accuracy with interpretability, making them valuable for scientific modeling. However, it is unclear a priori how deep a network needs to be for any given task, and deeper'

## Step 2-6: The cleaning function

We write one function, `clean_text`, that does steps 2-6 in order. Writing it as a function (rather than one giant blob of code) means we can easily test it on a single example before running it across all 30,000 rows -- much faster to debug.

In [4]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()


def strip_latex(text: str) -> str:
    """Remove common LaTeX/math artifacts found in arXiv abstracts.

    Order matters here: we remove $...$ math blocks FIRST (greedy removal of
    the whole formula), THEN clean up any remaining backslash commands that
    weren't inside $ $ (e.g. \\emph{...}).
    """
    text = re.sub(r"\$.*?\$", " ", text)          # remove $...$ inline math
    text = re.sub(r"\\[a-zA-Z]+\{.*?\}", " ", text)  # remove \command{...}
    text = re.sub(r"\\[a-zA-Z]+", " ", text)          # remove remaining \command
    return text


def clean_text(text: str) -> str:
    """Full cleaning pipeline: LaTeX removal -> lowercase -> remove non-letters
    -> tokenize -> remove stopwords -> lemmatize -> rejoin.
    """
    text = strip_latex(text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)  # keep only letters and whitespace
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]  # drop stopwords + very short tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

**Note on `len(t) > 2`:** this drops 1-2 letter tokens (leftover fragments, single letters from removed LaTeX). This is a judgment call worth documenting -- it slightly risks dropping legitimate short acronyms, but in practice greatly reduces noise from cleanup artifacts.

## Step 7: Sanity check on ONE example before running on all 30,000 rows

Always test on a small sample first. If the output looks wrong here, it's much faster to fix than after running on the full dataset.

In [5]:
sample_raw = df["text"].iloc[0]
sample_clean = clean_text(sample_raw)

print("BEFORE:\n", sample_raw[:400])
print("\nAFTER:\n", sample_clean[:400])

BEFORE:
 Multi-Exit Kolmogorov-Arnold Networks: enhancing accuracy and parsimony. Kolmogorov-Arnold Networks (KANs) uniquely combine high accuracy with interpretability, making them valuable for scientific modeling. However, it is unclear a priori how deep a network needs to be for any given task, and deeper KANs can be difficult to optimize and interpret. Here we introduce multi-exit KANs, where each laye

AFTER:
 multi exit kolmogorov arnold network enhancing accuracy parsimony kolmogorov arnold network kans uniquely combine high accuracy interpretability making valuable scientific modeling however unclear priori deep network need given task deeper kans difficult optimize interpret introduce multi exit kans layer includes prediction branch enabling network make accurate prediction multiple depth simultaneo


**Look at the AFTER output above.** Check: did LaTeX symbols disappear? Are there stray artifacts? Does it still read as recognizably about the same topic, just stripped down? If something looks off, fix `clean_text` here before moving on -- don't run on all 30,000 rows with a broken function.

## Step 7b: Run on the full dataset

Once the sample looks right, apply it to every row. With 30,000 rows this should take roughly 1-3 minutes -- `.apply()` runs the function once per row, so it scales linearly with dataset size.

In [6]:
df["clean_text"] = df["text"].apply(clean_text)
df[["text", "clean_text"]].head()

,text,clean_text
0,Multi-Exit Kolmogorov-Arnold Networks: enhanci...,multi exit kolmogorov arnold network enhancing...
1,LEGO: Spatial Accelerator Generation and Optim...,lego spatial accelerator generation optimizati...
2,MAVIS: Mathematical Visual Instruction Tuning ...,mavis mathematical visual instruction tuning a...
3,Topology-Aware Global-Local Mamba Networks for...,topology aware global local mamba network palm...
4,NTIRE 2025 challenge on Text to Image Generati...,ntire challenge text image generation model qu...


## Step 7c: Check for any empty results

Rarely, an abstract could clean down to nothing (e.g. if it was almost entirely math/LaTeX). Worth checking before Stage 4, since an empty document breaks TF-IDF.

In [7]:
empty_after_cleaning = (df["clean_text"].str.strip() == "").sum()
print("Rows that became empty after cleaning:", empty_after_cleaning)

if empty_after_cleaning > 0:
    df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)
    print("Dropped empty rows. New shape:", df.shape)

Rows that became empty after cleaning: 0


## Step 7d: Before/after vocabulary size comparison

A useful, quotable metric for your report: how much did cleaning reduce the vocabulary (unique word count)? Fewer, cleaner unique words means TF-IDF will build a more meaningful (less noisy) feature space in Stage 4.

In [10]:
raw_vocab = set(" ".join(df["text"].str.lower()).split())
clean_vocab = set(" ".join(df["clean_text"]).split())

print("Raw vocabulary size (rough, lowercase-only split):", len(raw_vocab))
print("Cleaned vocabulary size:", len(clean_vocab))
print("Reduction: {:.1f}%".format(100 * (1 - len(clean_vocab) / len(raw_vocab))))

Raw vocabulary size (rough, lowercase-only split): 234396
Cleaned vocabulary size: 60147
Reduction: 74.3%


What I see: Cleaning reduced the vocabulary from 234,396 to 60,147 unique tokens — a 74.3% reduction. This is a substantial but expected drop: English stopwords alone account for a large share of raw token diversity (through inflected forms across 30,000 documents), and lemmatization further collapses variants like "networks"/"network" or "training"/"trains" into single base forms.
What it means: A ~74% reduction indicates the cleaning pipeline is doing meaningful normalization without being so aggressive that it collapses genuinely distinct terms together. This gives TF-IDF (Stage 4) a much more compact, less sparse vocabulary to build features from, which should produce a more computationally tractable and topically coherent feature space than working from raw text directly.


## Step 8: Save for Stage 4

In [9]:
df.to_parquet("../data/processed/arxiv_preprocessed.parquet")
df.shape

(30000, 7)